# 07 — PaddleOCR-VL-1.6 Base Benchmark — Google Colab Pro

Uses the official PaddleOCR Python API and **disables layout detection** so the VLM receives each UIT-HWDB line crop directly. This matches our line-crop → transcription task rather than the full-page document-parsing pipeline.

Official sources:
- PaddleOCR-VL-1.6 model: https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6
- PaddleOCR-VL usage docs: https://www.paddleocr.ai/latest/en/version3.x/pipeline_usage/PaddleOCR-VL.html

The v1.6 model architecture is officially described as fully compatible with v1.5. For text-only line crops we use `prompt_label='ocr'` and no layout stage.

## 0. Install official PaddleOCR-VL runtime

In [ ]:
%pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q -U "paddleocr[doc-parser]>=3.6.0" "kagglehub>=1.0.2" "jiwer>=4.0.0" nvidia-ml-py
%pip install -q https://paddle-whl.bj.bcebos.com/nightly/cu126/safetensors/safetensors-0.6.2.dev0-cp38-abi3-linux_x86_64.whl
%pip install -q --force-reinstall opencv-python-headless "numpy==1.26.4"

> If Paddle/CUDA shared libraries fail immediately after installation, use **Runtime → Restart session** once, then continue from the next cell. The official PaddleOCR/ERNIE documentation recommends a CUDA 12.x environment and prefers its Docker image; Colab is a best-effort manual environment.

## 1. Data + evaluator

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

In [ ]:
def normalize_for_eval(text):
    # Strict OCR evaluation: Unicode NFC only.
    return unicodedata.normalize('NFC',str(text))

def compute_metrics(gt_list,pred_list):
    if len(gt_list)!=len(pred_list) or len(gt_list)==0:
        raise ValueError('GT/prediction lists must have the same non-zero length.')
    gt=[normalize_for_eval(x) for x in gt_list]
    pred=[normalize_for_eval(x) for x in pred_list]
    exact=sum(g==p for g,p in zip(gt,pred))/len(gt)
    return {'CER':float(cer(gt,pred)),'WER':float(wer(gt,pred)),'Exact_Line_Accuracy':float(exact),'N':int(len(gt))}
assert compute_metrics(['Việt Nam'],['Việt Nam'])['CER']==0.0
assert compute_metrics(['Biển Đông.'],['Biển đông.'])['CER']>0.0
SMOKE_SIZE=20
smoke_df=val_df.sample(n=SMOKE_SIZE,random_state=SEED).sort_index().reset_index(drop=True)
print('✅ Strict evaluator + fixed 20-sample validation smoke set ready.')

## 2. Build PaddleOCR-VL-1.6 pipeline

In [ ]:
import paddle
from paddleocr import PaddleOCRVL
assert paddle.device.is_compiled_with_cuda(),'Paddle CUDA build is not active.'
print('Paddle:',paddle.__version__); print('device:',paddle.device.get_device())
MODEL_NAME='PaddleOCR-VL-1.6'; MAX_NEW_TOKENS=256
RESULT_DIR=PROJECT_ROOT/'results'/'paddleocr_vl_1_6'/'base'; RESULT_DIR.mkdir(parents=True,exist_ok=True); smoke_df.to_csv(RESULT_DIR/'smoke_val_20.csv',index=False)
pipeline=PaddleOCRVL(pipeline_version='v1.6',use_layout_detection=False,use_doc_orientation_classify=False,use_doc_unwarping=False,device='gpu:0')
print('✅ pipeline initialized')

## 3. NVML memory monitor + result extraction

In [ ]:
import threading,time
from pynvml import nvmlInit,nvmlDeviceGetHandleByIndex,nvmlDeviceGetMemoryInfo
nvmlInit(); _nvml_handle=nvmlDeviceGetHandleByIndex(0)

def extract_text(result_obj):
    payload=result_obj.json
    if callable(payload): payload=payload()
    if isinstance(payload,dict) and isinstance(payload.get('res'),dict): payload=payload['res']
    blocks=payload.get('parsing_res_list',[]) if isinstance(payload,dict) else []
    texts=[str(b.get('block_content','')) for b in blocks if str(b.get('block_content','')).strip()]
    if texts: return '\n'.join(texts).strip()
    # Diagnostic fallback for API-shape changes.
    if isinstance(payload,dict):
        for key in ['rec_text','text','content','markdown_text']:
            if key in payload and payload[key] is not None: return str(payload[key]).strip()
    raise RuntimeError(f'Could not extract OCR text. Result keys: {list(payload) if isinstance(payload,dict) else type(payload)}')

def predict_one(image_path):
    samples=[]; stop=threading.Event()
    def monitor():
        while not stop.is_set():
            samples.append(nvmlDeviceGetMemoryInfo(_nvml_handle).used); time.sleep(0.03)
    t=threading.Thread(target=monitor,daemon=True); t.start(); t0=time.perf_counter()
    try:
        results=list(pipeline.predict(str(image_path),use_layout_detection=False,prompt_label='ocr',use_queues=False,max_new_tokens=MAX_NEW_TOKENS))
    finally:
        latency=time.perf_counter()-t0; stop.set(); t.join(timeout=1)
    assert len(results)==1,f'Expected one result, got {len(results)}'
    pred=extract_text(results[0]); peak=max(samples)/1024**3 if samples else None
    return pred,latency,peak

def run_benchmark(frame,split_name):
    rows=[]; started=time.perf_counter(); peaks=[]
    for i,(_,row) in enumerate(frame.iterrows(),1):
        pred,lat,peak=predict_one(resolve_image_path(row)); gt=normalize_for_eval(row['text']); pred=normalize_for_eval(pred); peaks.append(peak or 0)
        rows.append({'model':MODEL_NAME,'split':split_name,'writer_id':int(row['writer_id']),'filename':row['filename'],'relative_path':row['relative_path'],'ground_truth':gt,'prediction':pred,'sample_CER':float(cer(gt,pred)),'sample_WER':float(wer(gt,pred)),'exact_match':bool(gt==pred),'latency_sec':float(lat),'nvml_peak_used_gb':peak})
        if i<=20 or i%50==0 or i==len(frame): print(f'[{i}/{len(frame)}] CER={rows[-1]["sample_CER"]:.3f} | {lat:.3f}s')
    out=pd.DataFrame(rows); m=compute_metrics(out.ground_truth.tolist(),out.prediction.tolist()); m.update({'model':MODEL_NAME,'split':split_name,'total_runtime_sec':float(time.perf_counter()-started),'latency_mean_sec':float(out.latency_sec.mean()),'latency_p50_sec':float(out.latency_sec.quantile(.5)),'latency_p90_sec':float(out.latency_sec.quantile(.9)),'peak_gpu_memory_used_gb_nvml':float(max(peaks)) if peaks else None,'prompt_label':'ocr','max_new_tokens':MAX_NEW_TOKENS,'seed':SEED})
    return out,m

## 4. Single sample

In [ ]:
row=smoke_df.iloc[0]; pred,lat,peak=predict_one(resolve_image_path(row)); print('GT:',row['text']); print('PRED:',pred); print('latency=',lat,'peak used GB=',peak,'CER=',cer(normalize_for_eval(row['text']),normalize_for_eval(pred)))

## 5. 20-sample smoke

In [ ]:
preds,m=run_benchmark(smoke_df,'validation_smoke20'); display(preds[['writer_id','filename','ground_truth','prediction','sample_CER','latency_sec']]); print(json.dumps(m,ensure_ascii=False,indent=2)); preds.to_csv(RESULT_DIR/'paddle_base_smoke20_predictions.csv',index=False); (RESULT_DIR/'paddle_base_smoke20_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8')

## 6. Full validation gate

In [ ]:
RUN_FULL_VALIDATION=False
if RUN_FULL_VALIDATION:
    preds,m=run_benchmark(val_df.reset_index(drop=True),'validation'); preds.to_csv(RESULT_DIR/'paddle_base_val_predictions.csv',index=False); (RESULT_DIR/'paddle_base_val_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); print(json.dumps(m,ensure_ascii=False,indent=2))
else: print('Inspect smoke first.')

## 7. Frozen test gate

In [ ]:
RUN_TEST_BASELINE=False
if RUN_TEST_BASELINE:
    preds,m=run_benchmark(test_df.reset_index(drop=True),'test'); preds.to_csv(RESULT_DIR/'paddle_base_test_predictions.csv',index=False); (RESULT_DIR/'paddle_base_test_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); print(json.dumps(m,ensure_ascii=False,indent=2))
else: print('Frozen test untouched.')